# 02 — Pozos: limpieza del maestro de pozos (Capítulo IV)

**Objetivo:** aplicar las decisiones de limpieza tomadas sobre los hallazgos del perfilado y generar una versión limpia del archivo,
**sin modificar el original**.

| Dato | Valor |
| --- | --- |
| Archivo de origen | `data/raw/capitulo-iv-pozos.csv` (no se modifica) |
| Diagnóstico | `01_pozos_perfilado.ipynb`: hallazgos **H-01 a H-27** y decisiones **D-1 a D-12** |
| Diccionario | `docs/diccionario_pozos.csv` (v1.5): se usa para validar el resultado |
| Política aplicada | `POLITICA_CALIDAD_DATOS.md`: pasos 3 a 6 (elegir la acción, aplicar y medir, validar, documentar) |
| Resultado | `data/clean/capitulo-iv-pozos_limpio.csv` |

**Trazabilidad:** cada regla **R-xx** indica qué decisión (D-xx) aplica y qué hallazgos (H-xx) resuelve, y mide cuántas filas afectó.
Al final, el registro de reglas reúne todo el **linaje**: del archivo raw al archivo limpio, paso por paso.

**No se elimina ninguna fila.** Cuando hay duda, la política dice *marcar antes que eliminar*: las marcas son columnas nuevas de tipo sí/no.

## 1. Preparar el entorno

Importamos las librerías y definimos las rutas de los tres archivos que usa el notebook.
Mostramos la versión de pandas porque forma parte de la reproducibilidad.

- **hashlib** calcula la huella digital de un archivo (ver sección 2).
- **json** y **struct** leen y escriben las coordenadas, y **math** calcula distancias (regla de coordenadas).
- **os** mide el tamaño del archivo guardado.

In [1]:
import hashlib
import json
import math
import os
import struct

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)
print("Versión de pandas:", pd.__version__)

RUTA_RAW = "../data/raw/capitulo-iv-pozos.csv"
RUTA_DICCIONARIO = "../docs/diccionario_pozos.csv"
RUTA_CLEAN = "../data/clean/capitulo-iv-pozos_limpio.csv"

Versión de pandas: 3.0.6


## 2. Origen: leer el raw y registrar su huella digital

La **huella digital (SHA-256)** es un código que se calcula a partir del contenido completo del archivo.
Si el archivo cambiara aunque sea en un solo carácter, la huella sería otra.
Registrarla permite **probar de qué archivo exacto salió la versión limpia** (trazabilidad),
y al final del notebook la volvemos a calcular para probar que el raw no se modificó (integridad).

Leemos el raw en `raw`, que no se toca, y trabajamos sobre una **copia**, `df`.

In [2]:
def huella_sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as archivo:
        for bloque in iter(lambda: archivo.read(1024 * 1024), b""):
            h.update(bloque)
    return h.hexdigest()

huella_raw = huella_sha256(RUTA_RAW)
raw = pd.read_csv(RUTA_RAW, low_memory=False)
df = raw.copy()

print("Huella SHA-256 del raw:", huella_raw)
print(f"Filas: {len(df):,}  |  Columnas: {df.shape[1]}")

Huella SHA-256 del raw: 44c466133c7881753d49204bebf8e0ff3d0960c92ec7d7c0c63868946fd78e87
Filas: 85,611  |  Columnas: 26


## 3. Registro de reglas

Cada regla se anota en una lista con: su código, la decisión que aplica, los hallazgos que resuelve,
la acción de la política (*Corregir*, *Marcar* o *Sin acción*), qué hace y **cuántas filas afectó**.
Al final, esta lista se convierte en la tabla de linaje.

In [3]:
registro = []
TOTAL_FILAS = len(df)

def registrar(regla, decision, hallazgos, accion, que_hace, filas):
    registro.append({
        "regla": regla, "decision": decision, "hallazgos": hallazgos,
        "accion": accion, "que_hace": que_hace, "filas_afectadas": int(filas),
    })
    print(f"{regla} ({decision}): {int(filas):,} filas afectadas ({filas / TOTAL_FILAS * 100:.2f}%)")

## 4. Reglas de limpieza

Las reglas se aplican **en este orden**, porque el orden cambia el resultado:

1. **Primero los espacios sobrantes**: si no, las demás reglas comparan textos sucios
   (en el perfilado vimos que los espacios escondían parte de H-17 y H-22).
2. Después, los nulos ocultos y los valores imposibles.
3. Las coordenadas se corrigen **antes** de calcular las ubicaciones compartidas, porque mover un punto puede cambiar con quién comparte ubicación.
4. Las marcas de pozo físico y duplicados van **al final**, con el texto limpio y las coordenadas corregidas.

### R-01 · Quitar espacios sobrantes (D-8, H-25)

Quitamos los espacios al principio y al final de cada texto, y reemplazamos los espacios dobles por uno solo.
Es un error de formato evidente: acción **Corregir**.
No se aplica a `geojson` ni a `geom`, que no son textos para leer sino ubicaciones codificadas.

In [4]:
columnas_texto = [c for c in df.select_dtypes("str").columns if c not in ("geojson", "geom")]

antes = df[columnas_texto].copy()
for col in columnas_texto:
    df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

filas_r01 = (antes.fillna("") != df[columnas_texto].fillna("")).any(axis=1)
registrar("R-01", "D-8", "H-25", "Corregir", "Quita espacios al principio, al final y dobles en los textos", filas_r01.sum())

R-01 (D-8): 5,904 filas afectadas (6.90%)


### R-02 · "No informado" pasa a nulo real (D-1, H-03, H-04, H-06)

En estas seis columnas, el texto **"No informado"** es un dato faltante disfrazado: `isna()` no lo cuenta como faltante.
Lo convertimos en una celda vacía (nulo real). **No se borra ninguna fila**: solo cambia el contenido de esas celdas.

`sub_tipo_recurso` no está en esta lista porque ahí "No informado" tiene dos significados (reglas R-03 y R-04).

In [5]:
columnas_no_informado = ["clasificacion", "subclasificacion", "tipo_recurso", "tipopozo", "tipoextraccion", "tipoestado"]

es_no_informado = df[columnas_no_informado] == "No informado"
print("Celdas 'No informado' por columna:")
print(es_no_informado.sum().to_string())

df[columnas_no_informado] = df[columnas_no_informado].mask(es_no_informado)
registrar("R-02", "D-1", "H-03, H-04, H-06", "Corregir", "'No informado' pasa a nulo en 6 columnas", es_no_informado.any(axis=1).sum())

Celdas 'No informado' por columna:
clasificacion       18199
subclasificacion    18199
tipo_recurso        21859
tipopozo               55
tipoextraccion         55
tipoestado             55
R-02 (D-1): 21,898 filas afectadas (25.58%)


### R-03 · Subtipo de recurso: "No informado" pasa a "NO APLICA" (D-1, H-05, H-20)

SHALE y TIGHT son tipos de recurso **no convencional**. Para un pozo CONVENCIONAL o SIN RESERVORIO, la pregunta no aplica:
ahí "No informado" significa **"no aplica"**, no "falta el dato".

La regla se aplica **solo** a las celdas que dicen "No informado". Así no se pisa el pozo CONVENCIONAL con subtipo TIGHT,
que se conserva y se marca en la regla R-17 (decisión D-12).

In [6]:
no_aplica = (df["sub_tipo_recurso"] == "No informado") & df["tipo_recurso"].isin(["CONVENCIONAL", "SIN RESERVORIO"])

df.loc[no_aplica, "sub_tipo_recurso"] = "NO APLICA"
registrar("R-03", "D-1", "H-05, H-20", "Corregir", "Subtipo 'No informado' pasa a 'NO APLICA' en pozos CONVENCIONAL o SIN RESERVORIO", no_aplica.sum())

R-03 (D-1): 58,740 filas afectadas (68.61%)


### R-04 · Subtipo de recurso: el resto de "No informado" pasa a nulo (D-1, H-05)

Los "No informado" que quedan son faltantes reales (pozos NO CONVENCIONAL sin subtipo)
o casos en los que no se puede saber, porque el tipo de recurso también falta o es NO DISCRIMINADO.

In [7]:
subtipo_faltante = df["sub_tipo_recurso"] == "No informado"

df.loc[subtipo_faltante, "sub_tipo_recurso"] = pd.NA
registrar("R-04", "D-1", "H-05", "Corregir", "El resto de los subtipos 'No informado' pasa a nulo", subtipo_faltante.sum())
print(df["sub_tipo_recurso"].value_counts(dropna=False).to_string())

R-04 (D-1): 21,868 filas afectadas (25.54%)
sub_tipo_recurso
NO APLICA    58740
NaN          21868
SHALE         3380
TIGHT         1623


### R-05 · Datos faltantes que se conservan como nulos (D-2, H-01, H-02, H-07)

`empresa`, `formacion` y la fecha de inicio de perforación ya son nulos reales (celdas vacías).
Se decidió **no completarlos**: no hay un valor defendible. Solo 23 empresas se podrían tomar del archivo de producción 2026,
y en ese archivo la empresa coincide con la del maestro en apenas el 90% de los pozos.

Decidir no tocar algo también es una decisión: queda registrada, con 0 filas afectadas.

In [8]:
print(df[["empresa", "formacion", "adjiv_fecha_inicio_perf"]].isna().sum().rename("nulos que se conservan").to_string())
registrar("R-05", "D-2", "H-01, H-02, H-07", "Sin acción", "Se conservan como nulos: no hay valor defendible para completarlos", 0)

empresa                      964
formacion                   2815
adjiv_fecha_inicio_perf    34004
R-05 (D-2): 0 filas afectadas (0.00%)


### R-06 · Profundidad 0 pasa a nulo (D-5, H-10)

Un pozo no puede medir 0 metros: el 0 es un **dato faltante disfrazado de número**, igual que "No informado" en R-02.
(El criterio de la política *cero no es nulo* vale para la producción, donde un pozo cerrado produce 0 de verdad; no para la profundidad.)

In [9]:
profundidad_cero = df["profundidad"] == 0

df.loc[profundidad_cero, "profundidad"] = pd.NA
registrar("R-06", "D-5", "H-10", "Corregir", "Profundidad 0 pasa a nulo", profundidad_cero.sum())

R-06 (D-5): 10,299 filas afectadas (12.03%)


### R-07 · Profundidad mayor a 10.000 m: nulo + marca (D-5, H-11)

Estos valores son físicamente imposibles. Divididos por 10 (o por 100 en el caso de 378.939) se parecen mucho a la profundidad
típica de su yacimiento: probablemente es un **error de coma decimal**. Pero es una **hipótesis** que la fuente no confirma,
así que no se corrige: el valor pasa a nulo, el original queda en el raw y la fila se marca con `profundidad_sospechosa`.

In [10]:
LIMITE_PROFUNDIDAD = 10_000  # metros; diccionario v1.2, valor_max de profundidad

profundidad_imposible = df["profundidad"] > LIMITE_PROFUNDIDAD
print(raw.loc[profundidad_imposible, ["idpozo", "sigla", "profundidad"]].to_string())

df["profundidad_sospechosa"] = profundidad_imposible
df.loc[profundidad_imposible, "profundidad"] = pd.NA
registrar("R-07", "D-5", "H-11", "Marcar", "Profundidad mayor a 10.000 m pasa a nulo y se marca profundidad_sospechosa", profundidad_imposible.sum())

       idpozo              sigla  profundidad
4722    10269   TA.CMA-1.CullN-1      11021.0
21214   69901       YPF.Md.PC-33      24223.0
29364   97069   YPF.SC.LP.a-2162      10600.0
34636  102326      YPF.SC.CG-441      20680.0
37495  105170      YPF.SC.CS-590      15273.0
41544  109212     YPF.Ch.E.a-669      27064.0
75735  156804  YEA.RN.EFO-152(d)     378939.0
R-07 (D-5): 7 filas afectadas (0.01%)


### R-08 · Cota 0 en tierra: se conserva y se marca (D-6, H-12)

Un pozo **costa afuera** está al nivel del mar: ahí la cota 0 es un dato real y no se toca
(yacimiento Magallanes y jurisdicción "Estado Nacional").

Un pozo **en tierra** con cota 0 es dudoso: algunos podrían estar cerca de la costa, otros casi seguro no.
Como no se puede distinguir, se conserva el valor y se marca con `cota_sospechosa`.

In [11]:
costa_afuera = (df["yacimiento"] == "MAGALLANES") | (df["provincia"] == "Estado Nacional")
cota_cero = df["cota"] == 0
print("Cota 0 costa afuera (se deja):", (cota_cero & costa_afuera).sum(),
      "| cota 0 en tierra (se marca):", (cota_cero & ~costa_afuera).sum())

cota_cero_en_tierra = cota_cero & ~costa_afuera
df["cota_sospechosa"] = cota_cero_en_tierra
registrar("R-08", "D-6", "H-12", "Marcar", "Cota 0 en tierra se conserva y se marca cota_sospechosa", cota_cero_en_tierra.sum())

Cota 0 costa afuera (se deja): 117 | cota 0 en tierra (se marca): 108
R-08 (D-6): 108 filas afectadas (0.13%)


### R-09 · Cota imposible: nulo + marca (D-6, H-12, H-24)

Dos valores son imposibles **en su contexto**:

- una cota **negativa** en un pozo en tierra de Mendoza, en una zona que está a unos 1.400 m;
- una cota de **5.543 m** en Neuquén, más alta que el volcán Domuyo (unos 4.700 m), el punto más alto de la provincia.

Pasan a nulo y se marcan. La hipótesis de la coma corrida en el 5.543 (554,341 m) queda documentada, pero no se aplica.

In [12]:
ALTURA_MAXIMA_NEUQUEN = 4700  # volcán Domuyo (aproximado), igual que en el notebook 01

cota_imposible = ((df["cota"] < 0) & ~costa_afuera) | ((df["provincia"] == "Neuquén") & (df["cota"] > ALTURA_MAXIMA_NEUQUEN))
print(raw.loc[cota_imposible, ["idpozo", "sigla", "provincia", "cota"]].to_string())

df.loc[cota_imposible, "cota_sospechosa"] = True
df.loc[cota_imposible, "cota"] = pd.NA
registrar("R-09", "D-6", "H-12, H-24", "Marcar", "Cota imposible pasa a nulo y se marca cota_sospechosa", cota_imposible.sum())

       idpozo            sigla provincia     cota
46093  114060   YPF.Md.NQV.x-1   Mendoza  -100.00
72974  153315  YPF.Nq.Cto1-527   Neuquén  5543.41
R-09 (D-6): 2 filas afectadas (0.00%)


### R-10 · Fechas: de texto a tipo fecha (D-7, H-15)

Las cuatro columnas de fechas vienen como texto, y así no se pueden ordenar, restar ni filtrar por año.
Las convertimos a tipo fecha. Comprobamos que **ninguna fecha se pierda** en la conversión:
la cantidad de valores no vacíos tiene que ser la misma antes y después.

In [13]:
columnas_fecha = ["adjiv_fecha_inicio_perf", "adjiv_fecha_fin_perf", "adjiv_fecha_inicio_term", "adjiv_fecha_fin_term"]

no_vacias_antes = df[columnas_fecha].notna().sum()
for col in columnas_fecha:
    df[col] = pd.to_datetime(df[col], format="%Y-%m-%d")
assert (df[columnas_fecha].notna().sum() == no_vacias_antes).all(), "Se perdió alguna fecha en la conversión"

print(df[columnas_fecha].dtypes.to_string())
registrar("R-10", "D-7", "H-15", "Corregir", "Las 4 columnas de fechas pasan de texto a tipo fecha", df[columnas_fecha].notna().any(axis=1).sum())

adjiv_fecha_inicio_perf    datetime64[us]
adjiv_fecha_fin_perf       datetime64[us]
adjiv_fecha_inicio_term    datetime64[us]
adjiv_fecha_fin_term       datetime64[us]


R-10 (D-7): 51,607 filas afectadas (60.28%)


### R-11 · Fecha 1900-01-01 pasa a nulo (D-7, H-13)

Nadie perforó un pozo el 1 de enero de 1900: es una **fecha de relleno**, otro dato faltante disfrazado
(después del texto "No informado" y del número 0).

In [14]:
es_fecha_relleno = df[columnas_fecha] == pd.Timestamp("1900-01-01")
print(es_fecha_relleno.sum().rename("fechas 1900-01-01").to_string())

df[columnas_fecha] = df[columnas_fecha].mask(es_fecha_relleno)
registrar("R-11", "D-7", "H-13", "Corregir", "La fecha de relleno 1900-01-01 pasa a nulo", es_fecha_relleno.any(axis=1).sum())

adjiv_fecha_inicio_perf    37
adjiv_fecha_fin_perf       33
adjiv_fecha_inicio_term    33
adjiv_fecha_fin_term       33
R-11 (D-7): 37 filas afectadas (0.04%)


### R-12 · Fecha de fin anterior a la de inicio: se conserva y se marca (D-7, H-14)

No sabemos cuál de las dos fechas está mal (la de inicio, la de fin o las dos).
Intercambiarlas sería suponer el error; borrarlas, perder datos que probablemente son correctos.
Se conservan y se marcan con `fechas_inconsistentes`, para que quien calcule duraciones pueda excluirlas.

In [15]:
fechas_inconsistentes = (
    (df["adjiv_fecha_fin_perf"] < df["adjiv_fecha_inicio_perf"])
    | (df["adjiv_fecha_fin_term"] < df["adjiv_fecha_inicio_term"])
)

df["fechas_inconsistentes"] = fechas_inconsistentes
registrar("R-12", "D-7", "H-14", "Marcar", "Fecha de fin anterior a la de inicio: se conserva y se marca fechas_inconsistentes", fechas_inconsistentes.sum())

R-12 (D-7): 3 filas afectadas (0.00%)


### R-13 · Coordenadas invertidas: se intercambian y se marcan (D-11, H-19)

Dos pozos tienen la longitud y la latitud invertidas: así como están, caen en la Antártida.
A diferencia de la hipótesis de la coma decimal, esta corrección **se verifica con un dato independiente**:
intercambiadas, las coordenadas caen en medio de los demás pozos del mismo yacimiento.

Primero leemos las coordenadas de `geojson` y detectamos los puntos fuera de la Argentina (la misma caja que en el notebook 01).

In [16]:
def coordenadas_de(texto_geojson):
    return json.loads(texto_geojson)["coordinates"]

coordenadas = df["geojson"].map(coordenadas_de)
longitud, latitud = coordenadas.str[0], coordenadas.str[1]

coordenadas_invertidas = ~(latitud.between(-56, -21.7) & longitud.between(-74, -53))
print("Pozos con coordenadas fuera de la Argentina:", coordenadas_invertidas.sum())

Pozos con coordenadas fuera de la Argentina: 2


**Verificación antes de corregir:** medimos a qué distancia (en km, aproximada) queda el punto intercambiado
del centro de su yacimiento, calculado con los demás pozos del mismo yacimiento.

In [17]:
for i in df.index[coordenadas_invertidas]:
    vecinos = df.index[(df["yacimiento"] == df.at[i, "yacimiento"]) & ~coordenadas_invertidas]
    lat_centro, lon_centro = latitud[vecinos].median(), longitud[vecinos].median()
    lat_nueva, lon_nueva = longitud[i], latitud[i]  # intercambiadas
    km = math.hypot((lat_nueva - lat_centro) * 111, (lon_nueva - lon_centro) * 111 * math.cos(math.radians(lat_centro)))
    print(f"{df.at[i, 'sigla']} ({df.at[i, 'yacimiento']}): queda a {km:.1f} km del centro de su yacimiento ({len(vecinos)} pozos)")

SJ.RN.LN-7 (LOMA NEGRA): queda a 0.3 km del centro de su yacimiento (83 pozos)
PBE.Nq.M-1063(d) (EL MANGRULLO): queda a 2.6 km del centro de su yacimiento (146 pozos)


Los dos puntos corregidos caen cerca del centro de su yacimiento: la corrección está verificada.

Ahora intercambiamos las coordenadas en las **dos** columnas de ubicación, para que sigan diciendo lo mismo:

- en `geojson` se reescribe el texto con `[longitud, latitud]` en el orden correcto;
- en `geom` (formato WKB) se conservan los primeros 9 bytes (tipo de geometría y sistema de coordenadas, SRID 4326)
  y se reescriben los dos números de la ubicación.

Después comprobamos que `geom` y `geojson` sigan coincidiendo en todas las filas.

In [18]:
def geojson_punto(lon, lat):
    return json.dumps({"type": "Point", "coordinates": [lon, lat]}, separators=(",", ":"))

def geom_punto(geom_original, lon, lat):
    cabecera = bytes.fromhex(geom_original)[:9]
    return (cabecera + struct.pack("<dd", lon, lat)).hex().upper()

def punto_desde_wkb(texto_hex):
    datos = bytes.fromhex(texto_hex)
    return list(struct.unpack("<dd", datos[9:25]))

for i in df.index[coordenadas_invertidas]:
    lon_correcta, lat_correcta = latitud[i], longitud[i]
    df.at[i, "geojson"] = geojson_punto(lon_correcta, lat_correcta)
    df.at[i, "geom"] = geom_punto(df.at[i, "geom"], lon_correcta, lat_correcta)

df["coordenadas_corregidas"] = coordenadas_invertidas
print(df.loc[coordenadas_invertidas, ["idpozo", "sigla", "geojson"]].to_string())

coordenadas = df["geojson"].map(coordenadas_de)
puntos_geom = df["geom"].map(punto_desde_wkb)
coinciden = ((coordenadas.str[0] - puntos_geom.str[0]).abs() < 1e-9) & ((coordenadas.str[1] - puntos_geom.str[1]).abs() < 1e-9)
print("geom coincide con geojson en", coinciden.sum(), "de", len(df), "filas")
assert coinciden.all()

registrar("R-13", "D-11", "H-19", "Corregir", "Se intercambian longitud y latitud (geojson y geom) y se marca coordenadas_corregidas", coordenadas_invertidas.sum())

       idpozo             sigla                                                 geojson
4598    10143        SJ.RN.LN-7  {"type":"Point","coordinates":[-67.642383,-38.756338]}
80590  162058  PBE.Nq.M-1063(d)    {"type":"Point","coordinates":[-69.41577,-38.59007]}


geom coincide con geojson en 85611 de 85611 filas
R-13 (D-11): 2 filas afectadas (0.00%)


### R-14 · Sigla normalizada: identificador del pozo físico (D-8, H-26)

La sigla identifica al pozo físico, pero algunas se escriben de más de una forma (por ejemplo `NQ` y `Nq`).
La sigla original **no se modifica** (es un código oficial y sus minúsculas pueden tener significado).
Agregamos una columna nueva, `sigla_normalizada`, en mayúsculas (los espacios ya se quitaron en R-01).
La cantidad de filas afectadas son las que tienen una sigla escrita de más de una forma.

In [19]:
df["sigla_normalizada"] = df["sigla"].str.upper()

sigla_con_variantes = df.groupby("sigla_normalizada")["sigla"].transform("nunique") > 1
print("Pozos físicos (sigla normalizada):", df["sigla_normalizada"].nunique(), "| registros (idpozo):", len(df))
registrar("R-14", "D-8", "H-26", "Marcar", "Nueva columna sigla_normalizada (mayúsculas) para identificar al pozo físico", sigla_con_variantes.sum())

Pozos físicos (sigla normalizada): 78299 | registros (idpozo): 85611
R-14 (D-8): 85 filas afectadas (0.10%)


### R-15 · Marca de pozo con varias formaciones (D-3, H-08)

Un mismo pozo físico que produce de varias formaciones tiene un `idpozo` por cada una: no es un error.
La columna `es_multiformacion` marca esas filas, para que se pueda contar **pozos físicos** (por `sigla_normalizada`)
o **registros pozo + formación** (por `idpozo`) sin confundirlos.

In [20]:
es_multiformacion = df["sigla_normalizada"].duplicated(keep=False)

df["es_multiformacion"] = es_multiformacion
registrar("R-15", "D-3", "H-08", "Marcar", "Nueva columna es_multiformacion: el pozo físico tiene más de un registro", es_multiformacion.sum())

R-15 (D-3): 13,514 filas afectadas (15.79%)


### R-16 · Marca de posible carga duplicada (D-3, H-23)

Si dos filas repiten la **misma sigla y la misma formación**, no hay una capa distinta que justifique el registro aparte:
parecen cargas duplicadas. Se marcan con `posible_duplicado` y **no se eliminan** (podrían tener producción asociada).

In [21]:
posible_duplicado = df.duplicated(["sigla_normalizada", "formacion"], keep=False)

df["posible_duplicado"] = posible_duplicado
registrar("R-16", "D-3", "H-23", "Marcar", "Nueva columna posible_duplicado: misma sigla y misma formación", posible_duplicado.sum())

R-16 (D-3): 369 filas afectadas (0.43%)


### R-17 · Marca de ubicación compartida entre pozos distintos (D-4, H-09)

Muchas filas comparten ubicación porque son el mismo pozo físico (ya marcado en R-15).
Esta marca es para el otro caso: **pozos físicos distintos en el mismo punto**, como los que se perforan
desde una misma plataforma marina o locación. Se calcula después de corregir las coordenadas (R-13).

In [22]:
pozos_fisicos_en_el_punto = df.groupby("geojson")["sigla_normalizada"].transform("nunique")
ubicacion_compartida = pozos_fisicos_en_el_punto > 1

df["ubicacion_compartida"] = ubicacion_compartida
print("Puntos con más de un pozo físico:", df.loc[ubicacion_compartida, "geojson"].nunique())
registrar("R-17", "D-4", "H-09", "Marcar", "Nueva columna ubicacion_compartida: pozos físicos distintos en el mismo punto", ubicacion_compartida.sum())

Puntos con más de un pozo físico: 434
R-17 (D-4): 1,063 filas afectadas (1.24%)


### R-18 · Marca de tipo y subtipo de recurso contradictorios (D-12, H-21)

Un pozo CONVENCIONAL tiene subtipo TIGHT, que solo corresponde a pozos no convencionales.
No hay evidencia para saber cuál de los dos campos está mal: se conservan los dos y se marca con `recurso_inconsistente`.

In [23]:
recurso_inconsistente = (df["tipo_recurso"] == "CONVENCIONAL") & df["sub_tipo_recurso"].isin(["SHALE", "TIGHT"])

df["recurso_inconsistente"] = recurso_inconsistente
print(df.loc[recurso_inconsistente, ["idpozo", "sigla", "tipo_recurso", "sub_tipo_recurso"]].to_string())
registrar("R-18", "D-12", "H-21", "Marcar", "Nueva columna recurso_inconsistente: se conservan tipo y subtipo", recurso_inconsistente.sum())

       idpozo           sigla  tipo_recurso sub_tipo_recurso
46406  114385  YPF.Md.NLE.e-4  CONVENCIONAL            TIGHT
R-18 (D-12): 1 filas afectadas (0.00%)


### R-19 y R-20 · Decisiones sin acción sobre el dato (D-9, D-10)

Dos decisiones concluyeron que **el dato está bien y lo que faltaba era documentación**. Quedan en el registro con 0 filas afectadas:

- **D-9 (H-17, H-22, H-27):** los códigos de área y de yacimiento son la referencia; los nombres son etiquetas.
  "POZOS SIN YACIMIENTO" es un valor válido (el pozo no pertenece a ningún yacimiento).
- **D-10 (H-18):** "Estado Nacional" es una jurisdicción válida (pozos costa afuera de jurisdicción nacional).

Las dos decisiones están documentadas en el diccionario (v1.4 y v1.5).

In [24]:
registrar("R-19", "D-9", "H-17, H-22, H-27", "Sin acción", "El código es la referencia; 'POZOS SIN YACIMIENTO' es válido. Documentado en el diccionario", 0)
registrar("R-20", "D-10", "H-18", "Sin acción", "'Estado Nacional' es una jurisdicción válida. Documentado en el diccionario", 0)

R-19 (D-9): 0 filas afectadas (0.00%)
R-20 (D-10): 0 filas afectadas (0.00%)


## 5. Registro de reglas: el linaje completo

Esta tabla resume, en orden, todo lo que se le hizo al dato entre el raw y el archivo limpio.
Es la respuesta a "¿de dónde sale este valor?" para cualquier celda del resultado.

In [25]:
registro_reglas = pd.DataFrame(registro)
registro_reglas["pct"] = (registro_reglas["filas_afectadas"] / TOTAL_FILAS * 100).round(2)
registro_reglas

,regla,decision,hallazgos,accion,que_hace,filas_afectadas,pct
0,R-01,D-8,H-25,Corregir,"Quita espacios al principio, al final y dobles en los textos",5904,6.90
1,R-02,D-1,"H-03, H-04, H-06",Corregir,'No informado' pasa a nulo en 6 columnas,21898,25.58
2,R-03,D-1,"H-05, H-20",Corregir,Subtipo 'No informado' pasa a 'NO APLICA' en pozos CONVENCIONAL o SIN RESERV...,58740,68.61
3,R-04,D-1,H-05,Corregir,El resto de los subtipos 'No informado' pasa a nulo,21868,25.54
4,R-05,D-2,"H-01, H-02, H-07",Sin acción,Se conservan como nulos: no hay valor defendible para completarlos,0,0.00
5,R-06,D-5,H-10,Corregir,Profundidad 0 pasa a nulo,10299,12.03
6,R-07,D-5,H-11,Marcar,Profundidad mayor a 10.000 m pasa a nulo y se marca profundidad_sospechosa,7,0.01
7,R-08,D-6,H-12,Marcar,Cota 0 en tierra se conserva y se marca cota_sospechosa,108,0.13
8,R-09,D-6,"H-12, H-24",Marcar,Cota imposible pasa a nulo y se marca cota_sospechosa,2,0.00
9,R-10,D-7,H-15,Corregir,Las 4 columnas de fechas pasan de texto a tipo fecha,51607,60.28


## 6. Validación

Antes de guardar, comprobamos que el resultado cumple lo que prometimos (paso 5 de la política: *validar*).

### 6.1 Contra el diccionario de datos

El diccionario (v1.2 en adelante) tiene los **límites duros** de cada columna en `valor_min` y `valor_max`.
Los leemos y contamos cuántos valores del archivo limpio quedan fuera de esos límites. Tiene que dar **0** en todas.
Así el diccionario funciona como un **contrato**: si un cambio futuro rompe una regla, esta celda lo detecta.

In [26]:
diccionario = pd.read_csv(RUTA_DICCIONARIO, dtype=str)
con_limites = diccionario[(diccionario["valor_min"] != "-") | (diccionario["valor_max"] != "-")]

def a_limite(texto, columna):
    if texto == "-":
        return None
    return pd.Timestamp(texto) if columna in columnas_fecha else float(texto)

resultados = []
for _, fila in con_limites.iterrows():
    col = fila["columna"]
    if col == "geojson":
        coords = df["geojson"].map(coordenadas_de)
        valores = {"geojson (longitud)": (coords.str[0], -74, -53), "geojson (latitud)": (coords.str[1], -56, -21.7)}
    else:
        valores = {col: (df[col], a_limite(fila["valor_min"], col), a_limite(fila["valor_max"], col))}
    for nombre, (serie, minimo, maximo) in valores.items():
        fuera = pd.Series(False, index=df.index)
        if minimo is not None:
            fuera |= serie < minimo
        if maximo is not None:
            fuera |= serie > maximo
        resultados.append({"columna": nombre, "valor_min": minimo, "valor_max": maximo,
                           "minimo_observado": serie.min(), "maximo_observado": serie.max(),
                           "fuera_de_rango": int(fuera.sum())})

validacion = pd.DataFrame(resultados)
validacion

,columna,valor_min,valor_max,minimo_observado,maximo_observado,fuera_de_rango
0,idpozo,1.0,None,212,167348,0
1,cota,-105.0,6961.0,0.0,2978.3,0
2,profundidad,1.0,10000.0,1.0,8687.0,0
3,adjiv_fecha_inicio_perf,1900-01-02 00:00:00,2026-09-23 00:00:00,1900-07-27 00:00:00,2026-05-06 00:00:00,0
4,adjiv_fecha_fin_perf,1900-01-02 00:00:00,2026-09-23 00:00:00,1902-01-01 00:00:00,2026-05-22 00:00:00,0
5,adjiv_fecha_inicio_term,1900-01-02 00:00:00,2026-09-23 00:00:00,1902-01-01 00:00:00,2026-06-01 00:00:00,0
6,adjiv_fecha_fin_term,1900-01-02 00:00:00,2026-09-23 00:00:00,1902-01-01 00:00:00,2026-06-12 00:00:00,0
7,geojson (longitud),-74,-53,-72.1059,-57.76039,0
8,geojson (latitud),-56,-21.7,-54.02424,-22.00255,0


In [27]:
assert validacion["fuera_de_rango"].sum() == 0, "Hay valores fuera de los límites del diccionario"
print("Validación contra el diccionario: OK, ningún valor fuera de los límites duros")

Validación contra el diccionario: OK, ningún valor fuera de los límites duros


### 6.2 Integridad del resultado

Comprobamos que:

- el archivo limpio tiene **las mismas filas** que el raw (no se eliminó ninguna) y `idpozo` sigue siendo único;
- conserva **todas las columnas originales**, en el mismo orden, más las columnas de marca nuevas;
- ya no queda ningún "No informado" ni ninguna fecha 1900-01-01.

In [28]:
columnas_nuevas = [c for c in df.columns if c not in raw.columns]

assert len(df) == len(raw), "Cambió la cantidad de filas"
assert df["idpozo"].is_unique, "idpozo dejó de ser único"
assert list(df.columns[: raw.shape[1]]) == list(raw.columns), "Cambiaron las columnas originales"
assert not (df.select_dtypes("str") == "No informado").any().any(), "Quedan valores 'No informado'"
assert not (df[columnas_fecha] == pd.Timestamp("1900-01-01")).any().any(), "Quedan fechas 1900-01-01"

columnas_marca = [c for c in columnas_nuevas if df[c].dtype == bool]

print(f"Filas: raw {len(raw):,} | limpio {len(df):,}")
print("Columnas originales:", raw.shape[1], "| columnas nuevas:", len(columnas_nuevas), columnas_nuevas)
df[columnas_marca].sum().rename("filas marcadas").to_frame()

Filas: raw 85,611 | limpio 85,611
Columnas originales: 26 | columnas nuevas: 9 ['profundidad_sospechosa', 'cota_sospechosa', 'fechas_inconsistentes', 'coordenadas_corregidas', 'sigla_normalizada', 'es_multiformacion', 'posible_duplicado', 'ubicacion_compartida', 'recurso_inconsistente']


,filas marcadas
profundidad_sospechosa,7
cota_sospechosa,110
fechas_inconsistentes,3
coordenadas_corregidas,2
es_multiformacion,13514
posible_duplicado,369
ubicacion_compartida,1063
recurso_inconsistente,1


## 7. Guardar el archivo limpio

Guardamos el resultado en `data/clean`, con un nombre que indica su origen.
Las marcas (sí/no) se guardan como `True` / `False` y las fechas con formato `AAAA-MM-DD`.

Después volvemos a calcular la huella del **raw**: si es la misma que en la sección 2, queda probado que el original no se modificó.
También calculamos la huella del archivo limpio, para registrarla en `data/clean/README.md`.

In [29]:
df.to_csv(RUTA_CLEAN, index=False)

assert huella_sha256(RUTA_RAW) == huella_raw, "¡El raw cambió!"
huella_clean = huella_sha256(RUTA_CLEAN)

print("Raw sin modificar: OK (misma huella que al inicio)")
print("Archivo limpio:", RUTA_CLEAN)
print(f"Tamaño: {os.path.getsize(RUTA_CLEAN) / 1024 / 1024:.1f} MB")
print("Huella SHA-256 del limpio:", huella_clean)

Raw sin modificar: OK (misma huella que al inicio)
Archivo limpio: ../data/clean/capitulo-iv-pozos_limpio.csv
Tamaño: 36.8 MB
Huella SHA-256 del limpio: 08afa03010a2f507db3b725bd4a6d866397edaec3b568f706aa3e38c19b7e760


## 8. Comprobación final: releer el archivo guardado

Leemos el archivo recién guardado, como lo haría cualquier persona que lo use, y comprobamos que tiene las mismas filas y columnas,
y que las marcas y los nulos se conservaron al pasar por el CSV.

In [30]:
limpio = pd.read_csv(RUTA_CLEAN, low_memory=False, parse_dates=columnas_fecha)

assert limpio.shape == df.shape, "Cambió la forma de la tabla al guardar"
assert (limpio[columnas_marca].dtypes == bool).all(), "Alguna marca no se leyó como sí/no"
assert (limpio[columnas_marca].sum() == df[columnas_marca].sum()).all(), "Cambió alguna marca al guardar"
assert (limpio.isna().sum() == df.isna().sum()).all(), "Cambió la cantidad de nulos al guardar"
print("Archivo releído:", limpio.shape, "| marcas y nulos idénticos a los del DataFrame: OK")

Archivo releído: (85611, 35) | marcas y nulos idénticos a los del DataFrame: OK


## 9. Conclusiones

- **No se eliminó ninguna fila:** las 85.611 del raw están en el archivo limpio.
- **Se corrigieron** los formatos evidentes (espacios, fechas como texto, coordenadas invertidas verificadas)
  y los **faltantes disfrazados**, que en esta fuente aparecen de cuatro formas: texto ("No informado"), número (0),
  fecha (1900-01-01) y etiqueta ("POZOS SIN YACIMIENTO", que en este caso es un valor válido).
- **Se marcaron** los casos dudosos en columnas nuevas, en lugar de corregirlos sin evidencia.
- **El resultado cumple los límites duros del diccionario**, y la huella digital prueba que el raw no se modificó.

**Salvedades** (datos que conviene confirmar con la fuente): las unidades de `cota` y `profundidad`; el significado de `adjiv`
y de las letras de la sigla; la hipótesis de la coma decimal en profundidad y cota; la explicación de los códigos terminados en "R"
(áreas revertidas); y el criterio de las 12 millas para la jurisdicción "Estado Nacional".

**Etapa siguiente:** cruzar el maestro limpio con los archivos de producción por `idpozo`.